In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"

dbutils.widgets.text("file_id", "")
dbutils.widgets.text("new_adf_run_id", "")



In [0]:
file_id = dbutils.widgets.get("file_id").strip()
new_adf_run_id = dbutils.widgets.get("new_adf_run_id").strip()

max_retries = 3

if not file_id:
    raise ValueError("file_id is required")

print("Recovery preflight")
print("File ID:", file_id)
print("New ADF run ID:", new_adf_run_id or "Not assigned")
print("Maximum retries:", max_retries)


In [0]:

control = (
    spark.table(f"{catalog}.etl.file_control")
    .filter(F.col("file_id") == file_id)
)

rows = control.collect()

if len(rows) != 1:
    raise ValueError(
        f"Expected exactly one file-control record, found {len(rows)}"
    )

fc = rows[0]

print("File:", fc["file_name"])
print("Current stage:", fc["processing_stage"])
print("Pipeline status:", fc["pipeline_status"])
print("Previous ADF run:", fc["adf_run_id"])
print("Retry count:", fc["retry_count"])

if fc["processing_stage"] not in ("BRONZE", "SILVER", "GOLD"):
    raise ValueError("File is not at a recoverable stage")

if fc["pipeline_status"] != "IN_PROGRESS":
    raise ValueError("File is not marked IN_PROGRESS")

if not fc["is_enabled"]:
    raise ValueError("File is disabled")

if (fc["retry_count"] or 0) >= max_retries:
    raise ValueError("Maximum retry count reached")

print("File-control preflight passed")


In [0]:

from pyspark.sql import functions as F

audit_df = (
    spark.table(f"{catalog}.etl.file_stage_audit")
    .filter(
        (F.col("file_id") == file_id) &
        (F.col("processing_stage") == fc["processing_stage"])
    )
    .orderBy(
        F.col("attempt_number").desc(),
        F.col("stage_start_ts").desc()
    )
)

audit_rows = audit_df.collect()

if not audit_rows:
    raise ValueError(
        "No audit history found. Manual reconciliation required."
    )

display(
    audit_df.select(
        "audit_id",
        "file_id",
        "adf_run_id",
        "processing_stage",
        "attempt_number",
        "stage_status",
        "stage_start_ts",
        "stage_end_ts"
    )
)

successful_attempts = [
    row for row in audit_rows
    if row["stage_status"] == "SUCCESS"
]

unfinished_attempts = [
    row for row in audit_rows
    if row["stage_status"] == "IN_PROGRESS"
]

if successful_attempts:
    raise ValueError(
        "A successful stage audit already exists. "
        "Reconciliation required before retry."
    )

if len(unfinished_attempts) != 1:
    raise ValueError(
        f"Expected one unfinished audit, found {len(unfinished_attempts)}"
    )

previous_audit = unfinished_attempts[0]

if previous_audit["adf_run_id"] != fc["adf_run_id"]:
    raise ValueError(
        "Audit owner does not match file-control owner"
    )

print("Previous audit ID:", previous_audit["audit_id"])
print("Attempt number:", previous_audit["attempt_number"])
print("Audit status:", previous_audit["stage_status"])
print("Audit preflight passed")


In [0]:

from pyspark.sql import functions as F

batch_id = fc["batch_id"]

bronze_batch = (
    spark.table(f"{catalog}.bronze.patients_raw")
    .filter(F.col("file_id") == file_id)
)

silver = spark.table(f"{catalog}.silver.DimPatients")

# Check source count and duplicate patient IDs.
source_count = bronze_batch.count()

duplicate_count = (
    bronze_batch.groupBy("patient_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check whether Silver already contains versions
# explicitly associated with this batch.
silver_batch_count = (
    silver.filter(F.col("BatchId") == batch_id)
    .count()
)

print("Batch ID:", batch_id)
print("Bronze source count:", source_count)
print("Duplicate source patient IDs:", duplicate_count)
print("Silver versions with current BatchId:", silver_batch_count)

if source_count == 0:
    raise ValueError("Bronze source is empty")

if duplicate_count > 0:
    raise ValueError("Duplicate patient IDs found in Bronze")

if silver_batch_count > 0:
    raise ValueError(
        "Silver already contains records from this batch. "
        "Partial processing reconciliation required."
    )

print("Preliminary Silver reconciliation passed")


In [0]:

from pyspark.sql import functions as F

source_patients = (
    bronze_batch
    .select("patient_id")
    .distinct()
)

silver_summary = (
    silver
    .groupBy("PatientId")
    .agg(
        F.count("*").alias("total_versions"),
        F.sum(
            F.when(F.col("IsActive") == True, 1).otherwise(0)
        ).alias("active_versions"),
        F.sum(
            F.when(
                (F.col("IsActive") == False) &
                (F.col("RowEndDate").isNull()),
                1
            ).otherwise(0)
        ).alias("invalid_closed_versions")
    )
)

patient_validation = (
    source_patients.alias("b")
    .join(
        silver_summary.alias("s"),
        F.col("b.patient_id") == F.col("s.PatientId"),
        "left"
    )
    .select(
        F.col("b.patient_id"),
        F.coalesce(
            F.col("s.total_versions"), F.lit(0)
        ).alias("total_versions"),
        F.coalesce(
            F.col("s.active_versions"), F.lit(0)
        ).alias("active_versions"),
        F.coalesce(
            F.col("s.invalid_closed_versions"), F.lit(0)
        ).alias("invalid_closed_versions")
    )
    .withColumn(
        "scd_status",
        F.when(
            (F.col("total_versions") == 0) &
            (F.col("active_versions") == 0),
            "NEW_PATIENT"
        )
        .when(
            (F.col("total_versions") > 0) &
            (F.col("active_versions") == 1) &
            (F.col("invalid_closed_versions") == 0),
            "VALID_EXISTING_PATIENT"
        )
        .otherwise("REQUIRES_MANUAL_RECONCILIATION")
    )
    .orderBy("patient_id")
)

display(patient_validation)

unsafe_count = patient_validation.filter(
    F.col("scd_status") == "REQUIRES_MANUAL_RECONCILIATION"
).count()

if unsafe_count > 0:
    raise RuntimeError(
        f"{unsafe_count} patients have inconsistent SCD history. "
        "Do not retry Silver."
    )

print("Patient-level SCD structural validation passed")


In [0]:

# Read-only patient-level reconciliation

patient_ids = [
    row["patient_id"]
    for row in source_patients.collect()
]

print("=== BATCH 7 BRONZE SOURCE ===")

display(
    bronze_batch
    .orderBy("patient_id")
)

print("=== EXISTING SILVER PATIENT HISTORY ===")

silver_history = (
    silver
    .filter(F.col("PatientId").isin(patient_ids))
    .orderBy("PatientId", "RowStartDate")
)

display(silver_history)

print("=== SCD HISTORY DATE CHECK ===")

history_check = (
    silver_history
    .withColumn(
        "next_start_date",
        F.lead("RowStartDate").over(
            __import__(
                "pyspark.sql.window",
                fromlist=["Window"]
            ).Window
            .partitionBy("PatientId")
            .orderBy("RowStartDate")
        )
    )
)

display(
    history_check.select(
        "PatientId",
        "RowStartDate",
        "RowEndDate",
        "IsActive",
        "next_start_date"
    )
)

print(" inspection completed")


In [0]:

# Read-only recovery metadata inspection

catalog = "dbw_healthcare_de_dev"

print("=== FILE CONTROL SCHEMA ===")
spark.table(
    f"{catalog}.etl.file_control"
).printSchema()

print("=== STAGE AUDIT SCHEMA ===")
spark.table(
    f"{catalog}.etl.file_stage_audit"
).printSchema()

print("=== CURRENT BATCH 7 FILE CONTROL ===")
display(
    spark.table(f"{catalog}.etl.file_control")
    .filter(F.col("file_id") == file_id)
)

print("=== CURRENT BATCH 7 AUDIT HISTORY ===")
display(
    spark.table(f"{catalog}.etl.file_stage_audit")
    .filter(F.col("file_id") == file_id)
    .orderBy("attempt_number")
)


In [0]:

from pyspark.sql import functions as F

# Get current file-control information
control_rows = (
    spark.table(f"{catalog}.etl.file_control")
    .filter(F.col("file_id") == file_id)
    .collect()
)

if len(control_rows) != 1:
    raise ValueError("Expected exactly one file-control record")

fc = control_rows[0]

old_adf_run_id = fc["adf_run_id"]
current_stage = fc["processing_stage"]

# Find the abandoned audit for this file and execution
audit_rows = (
    spark.table(f"{catalog}.etl.file_stage_audit")
    .filter(
        (F.col("file_id") == file_id) &
        (F.col("adf_run_id") == old_adf_run_id) &
        (F.col("processing_stage") == current_stage) &
        (F.col("stage_status") == "IN_PROGRESS")
    )
    .collect()
)

if len(audit_rows) != 1:
    raise ValueError(
        f"Expected one unfinished audit, found {len(audit_rows)}. "
        "Manual reconciliation required."
    )

old_audit = audit_rows[0]
old_audit_id = old_audit["audit_id"]
old_attempt_number = old_audit["attempt_number"]

if old_audit["stage_end_ts"] is not None:
    raise ValueError("Audit already has an end timestamp")

print("File ID:", file_id)
print("Previous ADF Run:", old_adf_run_id)
print("Processing Stage:", current_stage)
print("Old Audit ID:", old_audit_id)
print("Previous Attempt:", old_attempt_number)


In [0]:

# Execute only after confirming the original run is terminal
# and the failure reason has been reconciled.

failure_reason = (
    "Original Databricks execution failed while loading "
    "the Silver stored procedure because of a Unity Catalog "
    "USE SCHEMA permission error. SQLSTATE 38000. "
    "Reconciled during automated recovery preparation."
)

spark.sql("""
UPDATE dbw_healthcare_de_dev.etl.file_stage_audit
SET
    stage_status = 'FAILED',
    stage_end_ts = CURRENT_TIMESTAMP(),
    error_message = :failure_reason
WHERE audit_id = :old_audit_id
  AND file_id = :file_id
  AND adf_run_id = :old_adf_run_id
  AND processing_stage = :current_stage
  AND attempt_number = :old_attempt_number
  AND stage_status = 'IN_PROGRESS'
  AND stage_end_ts IS NULL
""", args={
    "failure_reason": failure_reason,
    "old_audit_id": old_audit_id,
    "file_id": file_id,
    "old_adf_run_id": old_adf_run_id,
    "current_stage": current_stage,
    "old_attempt_number": old_attempt_number
})

print("Audit update executed")


In [0]:

audit_after = (
    spark.table(f"{catalog}.etl.file_stage_audit")
    .filter(F.col("audit_id") == old_audit_id)
)

display(
    audit_after.select(
        "audit_id",
        "file_id",
        "processing_stage",
        "attempt_number",
        "stage_status",
        "stage_end_ts",
        "error_message"
    )
)

rows = audit_after.collect()

if len(rows) != 1 or rows[0]["stage_status"] != "FAILED":
    raise RuntimeError("Audit reconciliation verification failed")

print("Step 214 completed successfully")


In [0]:

from uuid import uuid4
from pyspark.sql import functions as F

# Generate a new execution identifier
new_adf_run_id = str(uuid4())

# Maximum permitted recovery attempts
max_retries = 3

# Read the current file-control record
control_rows = (
    spark.table(f"{catalog}.etl.file_control")
    .filter(F.col("file_id") == file_id)
    .collect()
)

if len(control_rows) != 1:
    raise ValueError("Expected exactly one file-control record")

fc = control_rows[0]

# Validate recovery eligibility
assert fc["adf_run_id"] == old_adf_run_id
assert fc["processing_stage"] == current_stage
assert fc["pipeline_status"] == "IN_PROGRESS"
assert fc["is_enabled"] is True
assert (fc["retry_count"] or 0) < max_retries

# Confirm original audit is closed
audit_rows = (
    spark.table(f"{catalog}.etl.file_stage_audit")
    .filter(
        (F.col("audit_id") == old_audit_id) &
        (F.col("stage_status") == "FAILED")
    )
    .collect()
)

if len(audit_rows) != 1:
    raise ValueError("Original failed audit not found")

print("Recovery eligibility: PASSED")
print("File ID:", file_id)
print("Stage:", current_stage)
print("Previous ADF Run:", old_adf_run_id)
print("New Recovery Run ID:", new_adf_run_id)
print("Current Retry Count:", fc["retry_count"])
print("Maximum Retries:", max_retries)


In [0]:

# Claim Batch 7 for recovery
# Do not rerun this cell after a successful claim.

claim_result = spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_claim_recovery_file(
    :file_id,
    :old_adf_run_id,
    :new_adf_run_id,
    :current_stage,
    :max_retries
)
""", args={
    "file_id": file_id,
    "old_adf_run_id": old_adf_run_id,
    "new_adf_run_id": new_adf_run_id,
    "current_stage": current_stage,
    "max_retries": max_retries
})

display(claim_result)


In [0]:

control_after = (
    spark.table(f"{catalog}.etl.file_control")
    .filter(F.col("file_id") == file_id)
)

display(
    control_after.select(
        "file_id",
        "batch_id",
        "processing_stage",
        "pipeline_status",
        "retry_count",
        "adf_run_id",
        "processing_start_ts",
        "last_updated_ts"
    )
)

rows = control_after.collect()

if len(rows) != 1:
    raise RuntimeError("File-control verification failed")

fc_after = rows[0]

assert fc_after["adf_run_id"] == new_adf_run_id
assert fc_after["retry_count"] == 1
assert fc_after["processing_stage"] == "SILVER"
assert fc_after["pipeline_status"] == "IN_PROGRESS"

print("Recovery ownership successfully transferred")
print("Recovery ID:", fc_after["adf_run_id"])
print("Retry Count:", fc_after["retry_count"])


In [0]:

from pyspark.sql import functions as F

audit_table = f"{catalog}.etl.file_stage_audit"

recovery_audits = (
    spark.table(audit_table)
    .filter(
        (F.col("file_id") == file_id) &
        (F.col("processing_stage") == current_stage)
    )
    .orderBy("attempt_number")
)

display(
    recovery_audits.select(
        "audit_id",
        "file_id",
        "adf_run_id",
        "processing_stage",
        "attempt_number",
        "stage_status",
        "stage_start_ts",
        "stage_end_ts"
    )
)

# Check whether this recovery execution already has an audit
existing_recovery = (
    recovery_audits
    .filter(F.col("adf_run_id") == new_adf_run_id)
    .collect()
)

if existing_recovery:
    raise ValueError(
        "An audit already exists for this recovery execution. "
        "Do not create a duplicate."
    )

# Validate previous attempt
previous_attempts = recovery_audits.collect()

if not previous_attempts:
    raise ValueError("No previous Silver audit found")

latest_attempt = max(
    row["attempt_number"] for row in previous_attempts
)

if latest_attempt != old_attempt_number:
    raise ValueError(
        "Unexpected audit attempt sequence. "
        "Manual review required."
    )

next_attempt_number = latest_attempt + 1

print("Audit validation: PASSED")
print("Previous attempt:", latest_attempt)
print("Next attempt:", next_attempt_number)
print("Recovery execution:", new_adf_run_id)


In [0]:

from uuid import uuid4
from pyspark.sql import functions as F

# Generate one audit ID for this recovery attempt
new_audit_id = str(uuid4())

spark.sql("""
CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
    'START',
    :audit_id,
    :file_id,
    :run_id,
    :stage,
    CAST(NULL AS BIGINT),
    CAST(NULL AS BIGINT),
    CAST(NULL AS BIGINT),
    CAST(NULL AS STRING)
)
""", args={
    "audit_id": new_audit_id,
    "file_id": file_id,
    "run_id": new_adf_run_id,
    "stage": current_stage
})

# Verify the new audit was created
new_audit = (
    spark.table(f"{catalog}.etl.file_stage_audit")
    .filter(
        (F.col("file_id") == file_id) &
        (F.col("adf_run_id") == new_adf_run_id) &
        (F.col("processing_stage") == current_stage)
    )
)

display(
    new_audit.select(
        "audit_id",
        "attempt_number",
        "stage_status",
        "adf_run_id"
    )
)

rows = new_audit.collect()

if len(rows) != 1:
    raise RuntimeError("Expected exactly one recovery audit")

assert rows[0]["attempt_number"] == 2
assert rows[0]["stage_status"] == "IN_PROGRESS"

new_audit_id = rows[0]["audit_id"]

print("Silver recovery audit created successfully")
print("Audit ID:", new_audit_id)
print("Attempt:", rows[0]["attempt_number"])


In [0]:

# Execute Silver recovery for the claimed file

result = spark.sql("""
CALL dbw_healthcare_de_dev.silver.usp_LoadDimPatients(
    :file_id
)
""", args={
    "file_id": file_id
})

display(result)
